In [1]:
from pyspark.sql import functions as F

df_silver_keyword_perf = (
    spark.read
        .format("delta")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Silver.Lakehouse/Tables/dbo/fact_keyword_performance")
)

print(
    "Silver keyword performance rows:",
    df_silver_keyword_perf.count()
)

print(
    "Silver keyword performance columns:",
    len(df_silver_keyword_perf.columns)
)

display(df_silver_keyword_perf.limit(10))

StatementMeta(, b16d6633-96a2-40a2-8220-f8a3da266bbf, 3, Finished, Available, Finished, False)

Silver keyword performance rows: 29998
Silver keyword performance columns: 12


SynapseWidget(Synapse.DataFrame, b0ad21fe-b5f6-4269-a19c-eb573f72d71d)

In [2]:
df_date = spark.read.table("dim_date")
df_country = spark.read.table("dim_country")
df_device = spark.read.table("dim_device")
df_keyword = spark.read.table("dim_keyword")

StatementMeta(, b16d6633-96a2-40a2-8220-f8a3da266bbf, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql import functions as F

df_date = spark.read.table("dim_date")
df_country = spark.read.table("dim_country")
df_device = spark.read.table("dim_device")
df_keyword = spark.read.table("dim_keyword")

print("Date:", df_date.count())
print("Country:", df_country.count())
print("Device:", df_device.count())
print("Keyword:", df_keyword.count())

StatementMeta(, b16d6633-96a2-40a2-8220-f8a3da266bbf, 5, Finished, Available, Finished, False)

Date: 365
Country: 10
Device: 3
Keyword: 2000


In [4]:
df_gold_keyword_perf = (
    df_silver_keyword_perf.alias("f")

    # Date
    .join(
        df_date.select("date_key", "date").alias("d"),
        F.col("f.event_date") == F.col("d.date"),
        "left"
    )

    # Country
    .join(
        df_country.select("country_key", "country_code").alias("c"),
        F.col("f.country_code") == F.col("c.country_code"),
        "left"
    )

    # Device
    .join(
        df_device.select("device_key", "device_type").alias("dv"),
        F.col("f.device_type") == F.col("dv.device_type"),
        "left"
    )

    # Keyword
    .join(
        df_keyword.select("keyword_key", "keyword_id").alias("k"),
        F.col("f.keyword_id") == F.col("k.keyword_id"),
        "left"
    )
)

StatementMeta(, b16d6633-96a2-40a2-8220-f8a3da266bbf, 6, Finished, Available, Finished, False)

In [5]:
df_gold_keyword_perf = df_gold_keyword_perf.select(
    F.col("f.keyword_performance_id"),
    F.col("d.date_key"),
    F.col("c.country_key"),
    F.col("dv.device_key"),
    F.col("k.keyword_key"),
    F.col("f.search_volume"),
    F.col("f.ranking_position"),
    F.col("f.clicks"),
    F.col("f.impressions"),
    F.col("f.ctr"),
    F.col("f.ranking_bucket"),
    F.col("f.updated_timestamp")
)

StatementMeta(, b16d6633-96a2-40a2-8220-f8a3da266bbf, 7, Finished, Available, Finished, False)

In [6]:
print(
    "Gold keyword performance rows:",
    df_gold_keyword_perf.count()
)

print(
    "Gold keyword performance columns:",
    len(df_gold_keyword_perf.columns)
)

display(df_gold_keyword_perf.limit(10))

StatementMeta(, b16d6633-96a2-40a2-8220-f8a3da266bbf, 8, Finished, Available, Finished, False)

Gold keyword performance rows: 29998
Gold keyword performance columns: 12


SynapseWidget(Synapse.DataFrame, 04ce9c18-cf87-42ae-b696-8860d67a744e)

In [7]:
id_columns = [
    "date_key",
    "country_key",
    "device_key",
    "keyword_key"
]

for col_name in id_columns:
    null_count = (
        df_gold_keyword_perf
        .filter(F.col(col_name).isNull())
        .count()
    )

    print(f"{col_name}: {null_count} nulls")

StatementMeta(, b16d6633-96a2-40a2-8220-f8a3da266bbf, 9, Finished, Available, Finished, False)

date_key: 0 nulls
country_key: 0 nulls
device_key: 0 nulls
keyword_key: 0 nulls


In [8]:
(
    df_gold_keyword_perf.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_keyword_performance")
)

StatementMeta(, b16d6633-96a2-40a2-8220-f8a3da266bbf, 10, Finished, Available, Finished, False)

In [9]:
df_gold_keyword_perf_check = spark.read.table(
    "fact_keyword_performance"
)

print(
    "Gold fact_keyword_performance rows:",
    df_gold_keyword_perf_check.count()
)

print(
    "Gold fact_keyword_performance columns:",
    len(df_gold_keyword_perf_check.columns)
)

print(
    "Duplicate keyword performance IDs:",
    df_gold_keyword_perf_check
        .groupBy("keyword_performance_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

display(
    df_gold_keyword_perf_check.limit(10)
)

StatementMeta(, b16d6633-96a2-40a2-8220-f8a3da266bbf, 11, Finished, Available, Finished, False)

Gold fact_keyword_performance rows: 29998
Gold fact_keyword_performance columns: 12
Duplicate keyword performance IDs: 0


SynapseWidget(Synapse.DataFrame, 763eee88-15e7-426b-ba4d-3ac19a09895b)